# 모델 비교 실험 노트북

베이스라인 파이프라인을 공용으로 두고, **설정 셀의 `MODEL_ID`만 바꿔가며** 후보 모델들의 zero-shot 성능을 고정 평가셋에서 비교한다.

**사용 순서**
1. 아래 설정 셀에서 `MODEL_ID` 등 실험 변수 수정
2. 전체 셀 실행 (Run All)
3. 결과가 `outputs/experiments.csv`에 한 줄 누적됨
4. **다른 모델로 바꿀 때는 커널 재시작 후 재실행** (8GB VRAM 잔여물 방지)

모델은 `scripts/download_models.py`가 프로젝트의 `models/` 폴더에 받아둔다 (노트북은 로드만, 다운로드 금지).

| MODEL_ID | LOAD_IN_4BIT |
|---|---|
| ./models/Qwen2-VL-2B-Instruct | False |
| ./models/Qwen2.5-VL-3B-Instruct | False |
| ./models/Qwen2.5-VL-7B-Instruct | **True (필수)** |

In [ ]:
# ===== 실험 설정 (여기만 수정) =====
MODEL_ID = "./models/Qwen2-VL-2B-Instruct"   # scripts/download_models.py가 받아둔 로컬 폴더
LOAD_IN_4BIT = False        # 7B급은 True 필수 (VRAM 8GB)
MAX_PIXELS = 640 * 480      # 이미지당 픽셀 상한 (해상도 실험 변수, 원본이 저해상도라 여유값)
MAX_NEW_TOKENS = 32         # 리스트 하나만 출력하면 되므로 짧게
EVAL_N = 300                # 고정 평가셋 크기
SEED = 42

DATA_DIR = "./snuaichallenge_data/"
SPLIT_PATH = "./splits/holdout_300.csv"
RESULTS_PATH = "./outputs/experiments.csv"

In [ ]:
import ast
import os
import time
from datetime import datetime

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"device: {device}, dtype: {dtype}")

## 고정 평가셋 (hold-out)

train에서 시드 고정으로 뽑은 평가셋. **최초 1회 생성 후 파일로 고정**되어 모든 모델이 같은 문제로 평가받는다.

⚠️ 이 평가셋의 Id들은 이후 **파인튜닝 학습 데이터에서 반드시 제외**할 것 (안 그러면 평가 점수가 오염됨).

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

os.makedirs(os.path.dirname(SPLIT_PATH), exist_ok=True)
if os.path.exists(SPLIT_PATH):
    eval_df = pd.read_csv(SPLIT_PATH)
    print(f"기존 평가셋 로드: {len(eval_df)}개")
else:
    eval_df = train_df.sample(n=EVAL_N, random_state=SEED).reset_index(drop=True)
    eval_df.to_csv(SPLIT_PATH, index=False)
    print(f"평가셋 신규 생성: {len(eval_df)}개 -> {SPLIT_PATH}")

# 참고 기준선: 항상 [1,2,3,4]로 찍었을 때의 정확도 (No_ordering 비율)
ref_acc = (eval_df["Answer"] == "[1, 2, 3, 4]").mean()
print(f"찍기 기준선(항상 [1,2,3,4]): {ref_acc:.3f}")

## 모델 로드

`AutoModelForImageTextToText`를 쓰므로 Qwen2-VL / Qwen2.5-VL 등이 같은 코드로 로드된다.
모델은 사전에 `scripts/download_models.py`로 캐시에 받아둔 상태여야 한다 (여기서는 로드만).

In [ ]:
quant_cfg = None
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

t0 = time.time()
# local_files_only=True: 캐시에 없으면 몇 시간 매달리는 대신 즉시 에러.
# 다운로드는 scripts/download_models.py 전담 (노트북은 로드만).
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map=device,
    quantization_config=quant_cfg,
    local_files_only=True,
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS, local_files_only=True)
print(f"모델 로드 완료: {MODEL_ID} ({time.time() - t0:.0f}초)")

## 공용 함수 (프롬프트 / 파싱 / 채점)

모델 간 공정 비교를 위해 이 부분은 전 모델 동일하게 고정. (베이스라인과 같은 로직)

In [ ]:
def get_prompt_message(row, image_dir):
    """4장의 프레임과 문장으로 프롬프트 메시지를 구성한다. (베이스라인 동일)"""
    img_files = [row["Input_1"], row["Input_2"], row["Input_3"], row["Input_4"]]
    content = []
    for i, img_file in enumerate(img_files):
        content.append({"type": "image", "image": os.path.join(image_dir, row["Id"], img_file)})
        content.append({"type": "text", "text": f"\nImage {i + 1}\n"})

    content.append({"type": "text", "text": (
        f'Thinking about the sentence: "{row["Sentence"]}"\n'
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )})
    return [{"role": "user", "content": content}]


def parse_model_output(output_text):
    """출력에서 순열을 추출해 제출 형식(각 이미지의 원래 위치)으로 역변환.
    반환: (제출형식 리스트, 파싱 성공 여부)"""
    try:
        s, e = output_text.find("["), output_text.rfind("]")
        if s != -1 and e != -1:
            result = ast.literal_eval(output_text[s : e + 1])
            if isinstance(result, list) and sorted(result) == [1, 2, 3, 4]:
                sub = [0] * 4
                for idx, img_num in enumerate(result):
                    sub[img_num - 1] = idx + 1
                return sub, True
    except (ValueError, SyntaxError):
        pass
    return [1, 2, 3, 4], False

## 평가 실행

정확도와 함께 샘플당 시간·최대 VRAM을 기록한다 (본선 "자원 효율성" 근거 자료).

In [ ]:
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

records = []
t_start = time.time()

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    messages = get_prompt_message(row, TRAIN_IMAGE_DIR)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    out_text = processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

    pred, parsed = parse_model_output(out_text)
    gt = ast.literal_eval(row["Answer"])
    records.append({
        "Id": row["Id"], "pred": str(pred), "gt": str(gt),
        "correct": pred == gt, "parsed": parsed, "raw": out_text[:80],
    })

elapsed = time.time() - t_start
res_df = pd.DataFrame(records)
peak_vram = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0

In [ ]:
# ===== 결과 요약 + experiments.csv 누적 기록 =====
summary = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "model_id": MODEL_ID,
    "load_4bit": LOAD_IN_4BIT,
    "max_pixels": MAX_PIXELS,
    "eval_n": len(res_df),
    "accuracy": round(res_df["correct"].mean(), 4),
    "parse_fail": int((~res_df["parsed"]).sum()),
    "sec_per_sample": round(elapsed / len(res_df), 2),
    "peak_vram_gb": round(peak_vram, 2),
}

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
pd.DataFrame([summary]).to_csv(
    RESULTS_PATH, mode="a", index=False, header=not os.path.exists(RESULTS_PATH)
)

print(f"정확도:      {summary['accuracy']:.3f}  (찍기 기준선 {ref_acc:.3f})")
print(f"파싱 실패:   {summary['parse_fail']}개")
print(f"샘플당 시간: {summary['sec_per_sample']}초")
print(f"최대 VRAM:   {summary['peak_vram_gb']} GB")
print(f"\n누적 기록 -> {RESULTS_PATH}")
pd.read_csv(RESULTS_PATH)

## 오답 분석

틀린 샘플의 문장을 확인해 오답 유형(문장 단서 부족 / 프레임 유사 / 파싱 실패)을 파악한다.
이 결과를 텍스트·이미지 EDA 담당에게 전달.

In [ ]:
wrong = res_df[~res_df["correct"]].merge(eval_df[["Id", "Sentence", "No_ordering"]], on="Id")
print(f"오답 {len(wrong)}개 / 전체 {len(res_df)}개")
wrong_path = f"./outputs/errors_{MODEL_ID.split('/')[-1]}.csv"
wrong.to_csv(wrong_path, index=False)
print(f"오답 상세 저장 -> {wrong_path}")
wrong[["Id", "pred", "gt", "parsed", "raw", "Sentence"]].head(10)

: 